<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/RED_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import imageio
from IPython.display import Video, display

# 1. Configuración de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Ejecutando en: {device}")

# 2. Cargar Dataset (Fashion-MNIST)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, transform=transform, download=True
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

# 3. Definición de las Redes Neutonales (DCGAN ligera)
latent_dim = 100

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        return img.view(img.size(0), 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        return self.model(img_flat)

# Inicializar redes y optimizadores
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Vector fijo de ruido para evaluar la evolución visual constante
fixed_noise = torch.randn(25, latent_dim, device=device)

# Carpeta para guardar frames
os.makedirs('frames', exist_ok=True)
frame_list = []

print("Entrenando la GAN...")

# 4. Bucle de Entrenamiento
epochs = 15
frame_count = 0

for epoch in range(epochs):
    for i, (imgs, _) in enumerate(dataloader):
        batch_size = imgs.size(0)

        # Etiquetas
        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        real_imgs = imgs.to(device)

        # ---------------------
        # Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()

        outputs_real = D(real_imgs)
        d_loss_real = criterion(outputs_real, real_labels)

        z = torch.randn(batch_size, latent_dim, device=device)
        fake_imgs = G(z)
        outputs_fake = D(fake_imgs.detach())
        d_loss_fake = criterion(outputs_fake, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_D.step()

        # ---------------------
        # Train Generator
        # ---------------------
        optimizer_G.zero_grad()

        outputs = D(fake_imgs)
        g_loss = criterion(outputs, real_labels)

        g_loss.backward()
        optimizer_G.step()

        # Capturar frame cada cierto número de pasos para crear un video fluido
        if i % 60 == 0:
            G.eval()
            with torch.no_grad():
                gen_imgs = G(fixed_noise).cpu().numpy()
            G.train()

            # Generar rejilla visual 5x5 atractiva
            fig, axes = plt.subplots(5, 5, figsize=(5, 5))
            fig.suptitle(f"Epoch {epoch+1}/{epochs}", fontsize=12, color='white')
            fig.patch.set_facecolor('black')

            for k in range(25):
                ax = axes[k // 5, k % 5]
                ax.imshow(gen_imgs[k, 0], cmap='inferno') # Color 'inferno' para impacto visual
                ax.axis('off')

            plt.subplots_adjust(wspace=0.05, hspace=0.05)
            frame_path = f"frames/frame_{frame_count:04d}.png"
            plt.savefig(frame_path, facecolor=fig.get_facecolor(), bbox_inches='tight')
            plt.close()

            frame_list.append(imageio.v2.imread(frame_path))
            frame_count += 1

    print(f"Época [{epoch+1}/{epochs}] completada | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

# 5. Generar Video MP4
video_filename = "gan_tiktok_demo.mp4"
imageio.mimsave(video_filename, frame_list, fps=12)

print(f"\n¡Video generado exitosamente: {video_filename}!")

# Mostrar reproductor en Colab
display(Video(video_filename, embed=True, width=400))

Ejecutando en: cuda


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 163kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.15MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.6MB/s]


Entrenando la GAN...
Época [1/15] completada | D Loss: 1.3356 | G Loss: 0.7476
Época [2/15] completada | D Loss: 1.3106 | G Loss: 0.8604
Época [3/15] completada | D Loss: 1.3331 | G Loss: 0.8467
Época [4/15] completada | D Loss: 1.3526 | G Loss: 0.8638
Época [5/15] completada | D Loss: 1.3700 | G Loss: 0.7866
Época [6/15] completada | D Loss: 1.3327 | G Loss: 0.7259
Época [7/15] completada | D Loss: 1.3546 | G Loss: 0.7908
Época [8/15] completada | D Loss: 1.4015 | G Loss: 0.8019
Época [9/15] completada | D Loss: 1.4100 | G Loss: 0.6984
Época [10/15] completada | D Loss: 1.3473 | G Loss: 0.7931
Época [11/15] completada | D Loss: 1.3700 | G Loss: 0.7264
Época [12/15] completada | D Loss: 1.3587 | G Loss: 0.7761
Época [13/15] completada | D Loss: 1.3680 | G Loss: 0.8037
Época [14/15] completada | D Loss: 1.4018 | G Loss: 0.7075
Época [15/15] completada | D Loss: 1.3860 | G Loss: 0.7737



¡Video generado exitosamente: gan_tiktok_demo.mp4!


In [2]:
from google.colab import files

files.download('gan_tiktok_demo.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>